# Explicabilidade BGE por unidade contextual

Comparação executável e uniforme entre sete representações:

1. mensagem individual (`canonical_message`);
2. quatro mensagens do mesmo interlocutor (`speaker_block_4`);
3. sliding windows cronológicas de 4, 5, 6, 7 e 8 mensagens.

Os resultados globais permanecem separados por split, modo, seed e política.
As seções finais oferecem amostras concretas de **todas** as representações,
nos modos `frozen_probe` e `retrained`, sem carregar os 118 milhões de n-grams
inteiros na memória.


## Como interpretar

- **Comprehensiveness:** queda do suporte à classe prevista ao remover as evidências; maior tende a ser melhor.
- **Sufficiency distance:** distância para a saída original usando somente as evidências; menor tende a ser melhor.
- **Massa aditiva:** soma dos impactos isolados; não substitui a perturbação conjunta.
- **k95:** menor conjunto que atinge 95% do maior comprehensiveness observado na conversa.
- **Frozen probe:** checkpoint canônico aplicado à nova unidade sem retreino.
- **Retrained:** Transformer treinado especificamente para aquela representação.


In [1]:
from pathlib import Path
import json
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

warnings.filterwarnings('ignore', category=FutureWarning)
sns.set_theme(style='whitegrid', context='notebook')
pd.set_option('display.max_columns', 120)
pd.set_option('display.max_colwidth', 220)

def find_repository_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / 'backend' / 'notebooks').is_dir():
            return candidate
    fallback = Path(r'C:/Users/Master/Desktop/Recod_ia/HorusEmbbedingPlusML')
    if fallback.exists():
        return fallback
    raise FileNotFoundError('Raiz do repositório HorusEmbbedingPlusML não encontrada.')

REPO_ROOT = find_repository_root()
default_root = REPO_ROOT / 'backend' / 'experiment_results' / 'transformer' / 'studies' / 'hierarchical_explainability'
HIER_STUDY_ROOT = Path(os.environ.get('HORUS_BGE_EXPLAINABILITY_ROOT', default_root)).resolve()
HIER_SPLIT_COLORS = {'test_internal': '#2563eb', 'validation': '#f97316'}
HIER_CATEGORY_COLORS = {'TP': '#16a34a', 'TN': '#2563eb', 'FP': '#dc2626', 'FN': '#9333ea'}

print(f'Repositório: {REPO_ROOT}')
print(f'Artefatos:   {HIER_STUDY_ROOT / "context_blocks" / "aggregates"}')


Repositório: /home/andrei.pinto/HorusEmbbedingPlusML
Artefatos:   /home/andrei.pinto/HorusEmbbedingPlusML/backend/experiment_results/transformer/studies/hierarchical_explainability/context_blocks/aggregates


## Comparação uniforme das sete representações

Todos os gráficos abaixo aplicam o mesmo protocolo a `canonical_message`,
`speaker_block_4` e às janelas cronológicas 4–8. `frozen_probe` mede apenas
compatibilidade com o checkpoint antigo; `retrained` mede modelos treinados
para a nova unidade contextual.


In [2]:
CONTEXT_ROOT = HIER_STUDY_ROOT / 'context_blocks'
CONTEXT_AGG = CONTEXT_ROOT / 'aggregates'

def read_context(name, columns=None, filters=None):
    path = CONTEXT_AGG / f'{name}.parquet'
    if not path.exists():
        print(f'Artefato contextual ausente: {path}')
        return pd.DataFrame()
    return pd.read_parquet(path, columns=columns, filters=filters)

def context_parquet_info(name):
    path = CONTEXT_AGG / f'{name}.parquet'
    if not path.exists():
        return {'artefato': name, 'linhas': 0, 'tamanho_gib': 0, 'status': 'ausente'}
    import pyarrow.parquet as pq
    metadata = pq.ParquetFile(path).metadata
    return {
        'artefato': name, 'linhas': metadata.num_rows,
        'tamanho_gib': path.stat().st_size / 1024**3, 'status': 'disponível',
    }

context_manifest_path = CONTEXT_ROOT / 'manifest.json'
CTX_MANIFEST = (
    json.loads(context_manifest_path.read_text(encoding='utf-8'))
    if context_manifest_path.exists() else {}
)
CTX_EXPECTED_SHARDS = int(CTX_MANIFEST.get('expected_shards', 32))
_CTX_SHARD_CACHE = {}

def locate_context_shard(*, sample_id, split, representation, model_mode):
    key = (str(sample_id), split, representation, model_mode)
    if key in _CTX_SHARD_CACHE:
        return _CTX_SHARD_CACHE[key]
    base = CONTEXT_ROOT / 'explanations' / representation / model_mode / split
    for shard_index in range(CTX_EXPECTED_SHARDS):
        shard = base / f'shard-{shard_index:05d}'
        path = shard / 'conversation_metrics.parquet'
        if not path.exists():
            continue
        found = pd.read_parquet(
            path, columns=['sample_id'], filters=[('sample_id', '==', str(sample_id))]
        )
        if not found.empty:
            _CTX_SHARD_CACHE[key] = shard
            return shard
    raise KeyError(f'Shard não encontrado para {key}')

def read_context_detail(
    name, *, sample_id, split, representation, model_mode, seed=42, columns=None,
):
    shard = locate_context_shard(
        sample_id=sample_id, split=split,
        representation=representation, model_mode=model_mode,
    )
    path = shard / f'{name}.parquet'
    return pd.read_parquet(path, columns=columns, filters=[
        ('sample_id', '==', str(sample_id)), ('seed', '==', int(seed)),
    ])

ctx_conversations = read_context('conversation_metrics')
ctx_fidelity = read_context('fidelity')
ctx_controls = read_context('random_controls')
ctx_stability = read_context('stability')
ctx_summary = read_context('metric_summary')
ctx_categories = read_context('category_summary')
ctx_points = read_context('quantity_points')
ctx_classification = read_context('classification_metrics')
ctx_agreement = read_context('canonical_agreement')
ctx_paired = read_context('paired_common_fidelity')

# unit_effects e span_effects são lidos sob demanda por sample_id. Em especial,
# span_effects possui mais de 100 milhões de linhas e nunca deve ser
# materializado integralmente no kernel do notebook.
CTX_UNIT_COLUMNS = [
    'sample_id', 'split', 'representation', 'model_mode', 'seed',
    'prediction_category', 'block_index', 'message_model_indices', 'text',
    'delta_logit', 'abs_delta_logit', 'class_support_delta', 'direction',
    'selected_absolute_influence', 'selected_decision_supporting',
]
CTX_SPAN_COLUMNS = [
    'sample_id', 'split', 'representation', 'model_mode', 'seed',
    'prediction_category', 'block_index', 'message_model_index', 'span_text',
    'ngram_size', 'delta_logit', 'abs_delta_logit', 'class_support_delta', 'direction',
]

CTX_REPRESENTATIONS = [
    'canonical_message', 'speaker_block_4',
    'chronological_window_4', 'chronological_window_5',
    'chronological_window_6', 'chronological_window_7',
    'chronological_window_8',
]
CTX_COLORS = {
    'canonical_message': '#111827', 'speaker_block_4': '#dc2626',
    'chronological_window_4': '#2563eb', 'chronological_window_5': '#0891b2',
    'chronological_window_6': '#16a34a', 'chronological_window_7': '#ca8a04',
    'chronological_window_8': '#7c3aed',
}
CTX_MODE_STYLES = {'frozen_probe': '--', 'retrained': '-'}
CTX_MODEL_MODE = 'retrained'
CTX_POPULATION = 'paired_common'
CTX_SEED = 42
CTX_RANKING_POLICY = 'decision_supporting'
CTX_TOP_K_BOXPLOT = 6

display(pd.DataFrame({
    'configuração': ['modo', 'população', 'seed', 'ranking'],
    'valor': [CTX_MODEL_MODE, CTX_POPULATION, CTX_SEED, CTX_RANKING_POLICY],
}))
display(pd.DataFrame([
    context_parquet_info(name) for name in (
        'conversation_metrics', 'unit_effects', 'span_effects', 'fidelity',
        'random_controls', 'stability', 'metric_summary', 'quantity_points',
    )
]).style.format({'linhas': '{:,.0f}', 'tamanho_gib': '{:.3f}'}))
print('unit_effects e span_effects: leitura filtrada por conversa nas seções de amostragem.')


,configuração,valor
0,modo,retrained
1,população,paired_common
2,seed,42
3,ranking,decision_supporting


,artefato,linhas,tamanho_gib,status
0,conversation_metrics,"103,866",0.005,disponível
1,unit_effects,"1,107,096",0.129,disponível
2,span_effects,"118,746,626",6.411,disponível
3,fidelity,"1,819,661",0.154,disponível
4,random_controls,"894,127",0.022,disponível
5,stability,"69,244",0.007,disponível
6,metric_summary,"149,346",0.007,disponível
7,quantity_points,"399,822",0.003,disponível


unit_effects e span_effects: leitura filtrada por conversa nas seções de amostragem.


### 1. Auditoria, cobertura, blocos parciais e truncamento

In [3]:
cache_rows = []
for representation in CTX_REPRESENTATIONS:
    for source_split in ('train', 'validation'):
        path = CONTEXT_ROOT / 'cache' / representation / f'{source_split}_blocks.parquet'
        if not path.exists():
            continue
        columns = [
            'sample_id', 'is_partial', 'was_truncated', 'token_count',
            'max_token_length', 'overlap_fraction_with_previous',
        ]
        view = pd.read_parquet(path, columns=columns)
        cache_rows.append({
            'representation': representation,
            'source_split': source_split,
            'conversations': view.sample_id.nunique(),
            'units': len(view),
            'partial_rate': view.is_partial.mean(),
            'truncation_rate': view.was_truncated.mean(),
            'tokens_median': view.token_count.median(),
            'tokens_max': view.token_count.max(),
            'encoder_max_tokens': view.max_token_length.max(),
            'overlap_medio_anterior': view.overlap_fraction_with_previous.mean(),
        })
ctx_cache_audit = pd.DataFrame(cache_rows)
if not ctx_cache_audit.empty:
    display(ctx_cache_audit.style.format({
        'partial_rate': '{:.2%}', 'truncation_rate': '{:.2%}',
        'tokens_median': '{:.0f}', 'tokens_max': '{:.0f}',
        'overlap_medio_anterior': '{:.2%}',
    }))
    fig, axes = plt.subplots(1, 2, figsize=(17, 5))
    sns.barplot(data=ctx_cache_audit, x='representation', y='units', hue='source_split', ax=axes[0])
    sns.barplot(data=ctx_cache_audit, x='representation', y='truncation_rate', hue='source_split', ax=axes[1])
    axes[0].set_title('Unidades geradas por representação'); axes[1].set_title('Taxa de truncamento BGE')
    for ax in axes: ax.tick_params(axis='x', rotation=30)
    plt.tight_layout(); plt.show()


,representation,source_split,conversations,units,partial_rate,truncation_rate,tokens_median,tokens_max,encoder_max_tokens,overlap_medio_anterior
0,canonical_message,train,6950,79563,0.00%,0.00%,36,767,8192,0.00%
1,canonical_message,validation,1083,22552,0.00%,0.00%,22,1249,8192,0.00%
2,speaker_block_4,train,6950,26074,42.63%,0.00%,111,1493,8192,0.00%
3,speaker_block_4,validation,1083,6406,24.74%,0.00%,82,1960,8192,0.00%
4,chronological_window_4,train,6950,61120,1.32%,0.00%,133,2009,8192,66.47%
5,chronological_window_4,validation,1083,19303,0.00%,0.00%,93,2261,8192,70.79%
6,chronological_window_5,train,6950,54988,1.49%,0.00%,164,2473,8192,69.89%
7,chronological_window_5,validation,1083,18223,0.02%,0.00%,116,2624,8192,75.25%
8,chronological_window_6,train,6950,48892,1.75%,0.00%,194,2922,8192,71.49%
9,chronological_window_6,validation,1083,17143,0.02%,0.00%,139,2821,8192,78.07%


### 2. Métricas preditivas e matrizes TP/TN/FP/FN

In [4]:
if not ctx_classification.empty:
    all_primary = ctx_classification[ctx_classification.seed.eq(CTX_SEED)]
    primary = all_primary[all_primary.model_mode.eq(CTX_MODEL_MODE)]
    display(primary.sort_values(['model_mode', 'split', 'representation']))
    metrics = ['f1_macro', 'pr_auc', 'recall_scam', 'precision_scam']
    fig, axes = plt.subplots(2, 2, figsize=(18, 10), sharex=True)
    for ax, metric in zip(axes.flat, metrics):
        sns.barplot(
            data=primary, x='representation', y=metric,
            hue='split', palette=HIER_SPLIT_COLORS, ax=ax,
        )
        ax.set_title(f'{metric} | {CTX_MODEL_MODE} | seed {CTX_SEED}')
        ax.tick_params(axis='x', rotation=30)
    plt.tight_layout(); plt.show()

    confusion = all_primary.melt(
        id_vars=['split', 'representation', 'model_mode'],
        value_vars=[name for name in ('TP', 'TN', 'FP', 'FN') if name in all_primary],
        var_name='categoria', value_name='quantidade',
    )
    g = sns.catplot(
        data=confusion, x='representation', y='quantidade', hue='categoria',
        col='split', row='model_mode', kind='bar', height=4, aspect=1.7,
        palette=HIER_CATEGORY_COLORS,
    )
    g.set_xticklabels(rotation=30); g.fig.suptitle('Matrizes de decisão por representação', y=1.02)


,split,representation,model_mode,seed,model_name,dataset_name,accuracy,f1_macro,f1_scam,precision_macro,recall_macro,precision_scam,recall_scam,roc_auc,pr_auc,TP,TN,FP,FN
3,test_internal,canonical_message,retrained,42,Transformer_bge_canonical_message_retrained,test_internal,0.9482,0.9479,0.9520,0.9491,0.9470,0.9395,0.9649,0.9789,0.9807,714,604,46,26
9,test_internal,chronological_window_4,retrained,42,Transformer_bge_chronological_window_4_retrained,test_internal,0.9302,0.9299,0.9349,0.9303,0.9295,0.9292,0.9405,0.9737,0.9740,696,597,53,44
15,test_internal,chronological_window_5,retrained,42,Transformer_bge_chronological_window_5_retrained,test_internal,0.9338,0.9336,0.9376,0.9334,0.9338,0.9414,0.9338,0.9739,0.9758,691,607,43,49
21,test_internal,chronological_window_6,retrained,42,Transformer_bge_chronological_window_6_retrained,test_internal,0.9324,0.9320,0.9369,0.9325,0.9316,0.9307,0.9432,0.9754,0.9768,698,598,52,42
27,test_internal,chronological_window_7,retrained,42,Transformer_bge_chronological_window_7_retrained,test_internal,0.9381,0.9378,0.9421,0.9381,0.9376,0.9383,0.9459,0.9805,0.9822,700,604,46,40
33,test_internal,chronological_window_8,retrained,42,Transformer_bge_chronological_window_8_retrained,test_internal,0.9295,0.9291,0.9341,0.9295,0.9288,0.9291,0.9392,0.9784,0.9794,695,597,53,45
39,test_internal,speaker_block_4,retrained,42,Transformer_bge_speaker_block_4_retrained,test_internal,0.9122,0.9119,0.9173,0.9118,0.9120,0.9198,0.9149,0.9696,0.9682,677,591,59,63
45,validation,canonical_message,retrained,42,Transformer_bge_canonical_message_retrained,validation,0.9806,0.9373,0.8852,0.9040,0.9785,0.8100,0.9759,0.9948,0.9281,81,981,19,2
51,validation,chronological_window_4,retrained,42,Transformer_bge_chronological_window_4_retrained,validation,0.9825,0.9427,0.8950,0.9123,0.9795,0.8265,0.9759,0.9946,0.9815,81,983,17,2
57,validation,chronological_window_5,retrained,42,Transformer_bge_chronological_window_5_retrained,validation,0.9852,0.9510,0.9101,0.9253,0.9810,0.8526,0.9759,0.9936,0.9671,81,986,14,2


### 3. Concordância dos probes congelados com o canônico

In [5]:
if not ctx_agreement.empty:
    frozen = ctx_agreement[
        ctx_agreement.model_mode.eq('frozen_probe') & ctx_agreement.seed.eq(CTX_SEED)
    ]
    fig, axes = plt.subplots(1, 2, figsize=(17, 5))
    sns.barplot(data=frozen, x='representation', y='prediction_agreement', hue='split', palette=HIER_SPLIT_COLORS, ax=axes[0])
    sns.barplot(data=frozen, x='representation', y='mean_abs_logit_shift', hue='split', palette=HIER_SPLIT_COLORS, ax=axes[1])
    axes[0].set_ylim(0, 1); axes[0].set_title('Concordância da classe com canonical_message')
    axes[1].set_title('Mudança absoluta média do logit')
    for ax in axes: ax.tick_params(axis='x', rotation=30)
    plt.tight_layout(); plt.show()


### 4–5. Comprehensiveness e sufficiency por Top-k

In [6]:
def plot_context_summary(metric, title, ylabel):
    if ctx_summary.empty:
        return
    view = ctx_summary[
        ctx_summary.metric.eq(metric)
        & ctx_summary.population.eq(CTX_POPULATION)
        & ctx_summary.model_mode.eq(CTX_MODEL_MODE)
        & ctx_summary.seed.eq(CTX_SEED)
        & ctx_summary.ranking_policy.eq(CTX_RANKING_POLICY)
    ]
    fig, axes = plt.subplots(1, 2, figsize=(19, 6), sharey=True)
    for ax, split in zip(axes, ('test_internal', 'validation')):
        for representation, group in view[view.split.eq(split)].groupby('representation'):
            group = group.sort_values('top_k')
            ax.plot(group.top_k, group['mean'], color=CTX_COLORS[representation], label=representation)
            ax.fill_between(group.top_k, group.ci95_low, group.ci95_high, color=CTX_COLORS[representation], alpha=.10)
        ax.set(title=f'{title} | {split}\n{CTX_MODEL_MODE} | {CTX_POPULATION} | seed {CTX_SEED}', xlabel='Top-k unidades', ylabel=ylabel)
        ax.grid(alpha=.25); ax.legend(fontsize=8)
    plt.tight_layout(); plt.show()

plot_context_summary('comprehensiveness_logit_aligned', 'Comprehensiveness', 'Queda alinhada do logit')
plot_context_summary('sufficiency_distance_logit', 'Sufficiency distance', '|logit original − logit suficiente|')


### 6–8. Curvas por unidades, mensagens únicas e fração da conversa

In [7]:
def plot_context_xaxis(x, xlabel):
    if ctx_fidelity.empty:
        return
    view = ctx_fidelity[
        ctx_fidelity.model_mode.eq(CTX_MODEL_MODE)
        & ctx_fidelity.seed.eq(CTX_SEED)
        & ctx_fidelity.ranking_policy.eq(CTX_RANKING_POLICY)
    ].copy()
    if x == 'coverage_fraction':
        view['x_bin'] = (view[x] * 20).round() / 20
        x_plot = 'x_bin'
    else:
        x_plot = x
    fig, axes = plt.subplots(2, 2, figsize=(20, 12), sharey='row')
    for column, split in enumerate(('test_internal', 'validation')):
        data = view[view.split.eq(split)]
        split_n = data.sample_id.nunique()
        for representation, group in data.groupby('representation'):
            aggregate = group.groupby(x_plot, as_index=False).agg(
                comprehensiveness=('comprehensiveness_logit_aligned', 'mean'),
                sufficiency=('sufficiency_distance_logit', 'mean'),
                n=('sample_id', 'nunique'),
            ).sort_values(x_plot)
            axes[0, column].plot(aggregate[x_plot], aggregate.comprehensiveness, color=CTX_COLORS[representation], label=representation)
            axes[1, column].plot(aggregate[x_plot], aggregate.sufficiency, color=CTX_COLORS[representation], label=representation)
        axes[0, column].set_title(f'Comprehensiveness | {split} | {CTX_MODEL_MODE} | seed {CTX_SEED} | N={split_n}')
        axes[1, column].set_title(f'Sufficiency distance | {split} | {CTX_MODEL_MODE} | seed {CTX_SEED} | N={split_n}')
        for ax in axes[:, column]: ax.set_xlabel(xlabel); ax.grid(alpha=.25); ax.legend(fontsize=7)
    axes[0, 0].set_ylabel('Queda alinhada do logit'); axes[1, 0].set_ylabel('Distância no logit')
    plt.tight_layout(); plt.show()

plot_context_xaxis('top_k', 'Unidades acumuladas')
plot_context_xaxis('unique_messages_covered', 'Mensagens originais únicas')
plot_context_xaxis('coverage_fraction', 'Fração da conversa (faixas de 5%)')


### 9–11. Boxplots globais e por TP/TN/FP/FN

In [8]:
if not ctx_fidelity.empty:
    view = ctx_fidelity[
        ctx_fidelity.model_mode.eq(CTX_MODEL_MODE)
        & ctx_fidelity.seed.eq(CTX_SEED)
        & ctx_fidelity.ranking_policy.eq(CTX_RANKING_POLICY)
        & ctx_fidelity.top_k.eq(CTX_TOP_K_BOXPLOT)
    ]
    fig, axes = plt.subplots(2, 2, figsize=(20, 12), sharey='row')
    for column, split in enumerate(('test_internal', 'validation')):
        data = view[view.split.eq(split)]
        sns.boxplot(data=data, x='representation', y='comprehensiveness_logit_aligned', palette=CTX_COLORS, showfliers=False, ax=axes[0, column])
        sns.boxplot(data=data, x='representation', y='sufficiency_distance_logit', palette=CTX_COLORS, showfliers=False, ax=axes[1, column])
        axes[0, column].set_title(f'Comprehensiveness | {split} | Top-{CTX_TOP_K_BOXPLOT}')
        axes[1, column].set_title(f'Sufficiency | {split} | Top-{CTX_TOP_K_BOXPLOT}')
        for ax in axes[:, column]: ax.tick_params(axis='x', rotation=30)
    plt.tight_layout(); plt.show()

    g = sns.catplot(
        data=view, x='representation', y='comprehensiveness_logit_aligned',
        hue='prediction_category', col='split', kind='box', showfliers=False,
        palette=HIER_CATEGORY_COLORS, height=5, aspect=1.7,
    )
    g.set_xticklabels(rotation=30); g.fig.suptitle('Comprehensiveness por TP/TN/FP/FN', y=1.03)


### 12. Quantidade ideal: k95 e k_joint

In [9]:
if not ctx_points.empty:
    reached = ctx_points[
        ctx_points.status.eq('reached')
        & ctx_points.model_mode.eq(CTX_MODEL_MODE)
        & ctx_points.seed.eq(CTX_SEED)
        & ctx_points.ranking_policy.eq(CTX_RANKING_POLICY)
    ]
    fig, axes = plt.subplots(2, 2, figsize=(20, 12))
    for column, point_type in enumerate(('k95', 'k_joint')):
        data = reached[reached.point_type.eq(point_type)]
        sns.boxplot(data=data, x='representation', y='unique_messages', hue='split', palette=HIER_SPLIT_COLORS, showfliers=False, ax=axes[0, column])
        sns.boxplot(data=data, x='representation', y='fraction', hue='split', palette=HIER_SPLIT_COLORS, showfliers=False, ax=axes[1, column])
        axes[0, column].set_title(f'{point_type}: mensagens originais únicas')
        axes[1, column].set_title(f'{point_type}: fração da conversa')
        axes[1, column].yaxis.set_major_formatter(lambda value, _: f'{100*value:.0f}%')
        for ax in axes[:, column]: ax.tick_params(axis='x', rotation=30)
    plt.tight_layout(); plt.show()
    display(reached.groupby(['split', 'representation', 'point_type']).agg(
        N=('sample_id', 'nunique'), units_mediana=('units', 'median'),
        mensagens_mediana=('unique_messages', 'median'), fracao_mediana=('fraction', 'median'),
    ))


N  units_mediana  \
split         representation         point_type                        
test_internal canonical_message      k95         1325            7.0   
                                     k_joint      837            9.0   
              chronological_window_4 k95         1304            7.0   
                                     k_joint     1118            7.0   
              chronological_window_5 k95         1297            6.0   
                                     k_joint     1180            7.0   
              chronological_window_6 k95         1307            6.0   
                                     k_joint     1230            6.0   
              chronological_window_7 k95         1303            5.0   
                                     k_joint     1255            5.0   
              chronological_window_8 k95         1302            4.0   
                                     k_joint     1273            4.0   
              speaker_block_4        k95         1280            3.0   
                                     k_joint     1041            4.0   
validation    canonical_message      k95         1083            9.0   
                                     k_joint      167            9.0   
              chronological_window_4 k95         1074            9.0   
                                     k_joint      783            9.0   
              chronological_window_5 k95         1066            9.0   
                                     k_joint      898            9.0   
              chronological_window_6 k95         1062            8.0   
                                     k_joint      944            8.0   
              chronological_window_7 k95         1068            8.0   
                                     k_joint      983            8.0   
              chronological_window_8 k95         1072            7.0   
                                     k_joint      996            7.0   
              speaker_block_4        k95         1045            2.0   
                                     k_joint      529            2.0   

                                                 mensagens_mediana  \
split         representation         point_type                      
test_internal canonical_message      k95                       7.0   
                                     k_joint                   9.0   
              chronological_window_4 k95                      10.0   
                                     k_joint                  11.0   
              chronological_window_5 k95                      10.0   
                                     k_joint                  11.0   
              chronological_window_6 k95                      11.0   
                                     k_joint                  11.0   
              chronological_window_7 k95                      11.0   
                                     k_joint                  11.0   
              chronological_window_8 k95                      11.0   
                                     k_joint                  11.0   
              speaker_block_4        k95                      10.0   
                                     k_joint                  11.0   
validation    canonical_message      k95                       9.0   
                                     k_joint                   9.0   
              chronological_window_4 k95                      14.0   
                                     k_joint                  14.0   
              chronological_window_5 k95                      14.0   
                                     k_joint                  14.0   
              chronological_window_6 k95                      14.0   
                                     k_joint                  14.0   
              chronological_window_7 k95                      14.0   
                                     k_joint                  14.0   
              chronological_window_8 k95                      14.0   
 

### 13. Massa aditiva, comprehensiveness e redundancy gap

In [10]:
if not ctx_fidelity.empty:
    mass = ctx_fidelity[
        ctx_fidelity.model_mode.eq(CTX_MODEL_MODE)
        & ctx_fidelity.seed.eq(CTX_SEED)
        & ctx_fidelity.ranking_policy.eq(CTX_RANKING_POLICY)
    ]
    fig, axes = plt.subplots(3, 2, figsize=(20, 16), sharex='col')
    metric_specs = [
        ('massa', 'Massa aditiva'),
        ('conjunto', 'Comprehensiveness conjunto'),
        ('redundancia', 'Redundancy gap'),
    ]
    for column, split in enumerate(('test_internal', 'validation')):
        data = mass[mass.split.eq(split)]
        for representation, group in data.groupby('representation'):
            aggregate = group.groupby('unique_messages_covered', as_index=False).agg(
                massa=('cumulative_evidence_mass_logit', 'mean'),
                conjunto=('comprehensiveness_logit_aligned', 'mean'),
                redundancia=('redundancy_gap_logit', 'mean'),
            )
            for row, (metric, _) in enumerate(metric_specs):
                axes[row, column].plot(
                    aggregate.unique_messages_covered, aggregate[metric],
                    color=CTX_COLORS[representation], label=representation,
                )
        for row, (_, title) in enumerate(metric_specs):
            axes[row, column].set_title(
                f'{title} | {split} | {CTX_MODEL_MODE} | seed {CTX_SEED} | N={data.sample_id.nunique()}'
            )
            axes[row, column].set_xlabel('Mensagens originais únicas cobertas')
            axes[row, column].set_ylabel('Logit')
            axes[row, column].grid(alpha=.25); axes[row, column].legend(fontsize=7)
    plt.tight_layout(); plt.show()


### 14. Trechos versus controles aleatórios

In [11]:
if not ctx_controls.empty:
    controls = ctx_controls[
        ctx_controls.model_mode.eq(CTX_MODEL_MODE) & ctx_controls.seed.eq(CTX_SEED)
    ].copy()
    controls['specificity_ratio'] = controls.selected_abs_delta_logit / controls.control_abs_delta_logit.replace(0, np.nan)
    impact = pd.concat([
        controls[['split', 'representation', 'selected_abs_delta_logit']].rename(columns={'selected_abs_delta_logit': 'impact'}).assign(tipo='Selecionado'),
        controls[['split', 'representation', 'control_abs_delta_logit']].rename(columns={'control_abs_delta_logit': 'impact'}).assign(tipo='Controle'),
    ])
    fig, axes = plt.subplots(1, 2, figsize=(20, 6))
    sns.boxplot(data=impact, x='representation', y='impact', hue='tipo', showfliers=False, ax=axes[0])
    sns.boxplot(data=controls, x='representation', y='specificity_ratio', hue='split', palette=HIER_SPLIT_COLORS, showfliers=False, ax=axes[1])
    axes[1].axhline(1, color='black', ls='--')
    for ax in axes: ax.tick_params(axis='x', rotation=30)
    axes[0].set_title('Impacto dos trechos versus controles'); axes[1].set_title('Razão de especificidade')
    plt.tight_layout(); plt.show()


### 15. Estabilidade entre seeds

In [12]:
if not ctx_stability.empty:
    view = ctx_stability[ctx_stability.model_mode.eq(CTX_MODEL_MODE)]
    fig, axes = plt.subplots(2, 3, figsize=(22, 12))
    stability_metrics = [
        ('top_k_jaccard', 'Jaccard Top-6'),
        ('direction_agreement', 'Concordância de direção'),
        ('rank_spearman', 'Spearman dos rankings'),
        ('rank_kendall', 'Kendall dos rankings'),
        ('impact_cv', 'Coeficiente de variação do impacto'),
    ]
    for ax, (metric, title) in zip(axes.flat, stability_metrics):
        sns.boxplot(
            data=view, x='representation', y=metric, hue='split',
            palette=HIER_SPLIT_COLORS, showfliers=False, ax=ax,
        )
        ax.set_title(title); ax.tick_params(axis='x', rotation=30)
    agreement = view.groupby(['split', 'representation'], as_index=False).prediction_agreement.mean()
    sns.barplot(data=agreement, x='representation', y='prediction_agreement', hue='split', palette=HIER_SPLIT_COLORS, ax=axes.flat[-1])
    axes.flat[-1].set_ylim(0, 1); axes.flat[-1].tick_params(axis='x', rotation=30)
    axes.flat[-1].set_title('Concordância da classe')
    plt.tight_layout(); plt.show()


### 16. Comparação pareada entre representações

In [13]:
if not ctx_paired.empty:
    paired_view = ctx_paired[
        ctx_paired.model_mode.eq(CTX_MODEL_MODE)
        & ctx_paired.seed.eq(CTX_SEED)
        & ctx_paired.ranking_policy.eq(CTX_RANKING_POLICY)
        & ctx_paired.top_k.eq(CTX_TOP_K_BOXPLOT)
    ]
    canonical = paired_view[paired_view.representation.eq('canonical_message')][
        ['split', 'sample_id', 'comprehensiveness_logit_aligned', 'sufficiency_distance_logit']
    ].rename(columns={
        'comprehensiveness_logit_aligned': 'canonical_comprehensiveness',
        'sufficiency_distance_logit': 'canonical_sufficiency',
    })
    comparison = paired_view.merge(canonical, on=['split', 'sample_id'])
    comparison['delta_comprehensiveness'] = comparison.comprehensiveness_logit_aligned - comparison.canonical_comprehensiveness
    comparison['delta_sufficiency'] = comparison.sufficiency_distance_logit - comparison.canonical_sufficiency
    fig, axes = plt.subplots(1, 2, figsize=(19, 6))
    sns.boxplot(data=comparison, x='representation', y='delta_comprehensiveness', hue='split', palette=HIER_SPLIT_COLORS, showfliers=False, ax=axes[0])
    sns.boxplot(data=comparison, x='representation', y='delta_sufficiency', hue='split', palette=HIER_SPLIT_COLORS, showfliers=False, ax=axes[1])
    for ax in axes: ax.axhline(0, color='black', ls='--'); ax.tick_params(axis='x', rotation=30)
    axes[0].set_title('Δ comprehensiveness versus canônico (maior é melhor)')
    axes[1].set_title('Δ sufficiency versus canônico (menor é melhor)')
    plt.tight_layout(); plt.show()


### 16.1. Resumo específico das janelas cronológicas 4–8

In [14]:
chronological = [f'chronological_window_{size}' for size in range(4, 9)]
if not ctx_classification.empty:
    predictive = ctx_classification[
        ctx_classification.representation.isin(chronological)
        & ctx_classification.seed.eq(CTX_SEED)
    ].copy()
    predictive['window_size'] = predictive.representation.str.rsplit('_', n=1).str[-1].astype(int)
    fig, axes = plt.subplots(1, 2, figsize=(18, 5))
    for mode in ('frozen_probe', 'retrained'):
        for split in ('test_internal', 'validation'):
            group = predictive[predictive.model_mode.eq(mode) & predictive.split.eq(split)].sort_values('window_size')
            label = f'{split} | {mode}'
            axes[0].plot(group.window_size, group.f1_macro, color=HIER_SPLIT_COLORS[split], ls=CTX_MODE_STYLES[mode], marker='o', label=label)
            axes[1].plot(group.window_size, group.pr_auc, color=HIER_SPLIT_COLORS[split], ls=CTX_MODE_STYLES[mode], marker='o', label=label)
    axes[0].set_title('F1 macro por tamanho da janela'); axes[1].set_title('PR-AUC por tamanho da janela')
    for ax in axes: ax.set_xlabel('n mensagens por janela'); ax.grid(alpha=.25); ax.legend(fontsize=8)
    plt.tight_layout(); plt.show()

if not ctx_points.empty:
    chronological_points = ctx_points[
        ctx_points.representation.isin(chronological)
        & ctx_points.model_mode.eq(CTX_MODEL_MODE)
        & ctx_points.seed.eq(CTX_SEED)
        & ctx_points.ranking_policy.eq(CTX_RANKING_POLICY)
        & ctx_points.status.eq('reached')
    ].copy()
    chronological_points['window_size'] = chronological_points.representation.str.rsplit('_', n=1).str[-1].astype(int)
    display(chronological_points.groupby(['split', 'window_size', 'point_type']).agg(
        N=('sample_id', 'nunique'), mensagens_mediana=('unique_messages', 'median'),
        fracao_mediana=('fraction', 'median'), redundancia_mediana=('redundancy_ratio', 'median'),
    ))


N  mensagens_mediana  fracao_mediana  \
split         window_size point_type                                            
test_internal 4           k95         1304               10.0        0.909091   
                          k_joint     1118               11.0        1.000000   
              5           k95         1297               10.0        0.937500   
                          k_joint     1180               11.0        1.000000   
              6           k95         1307               11.0        1.000000   
                          k_joint     1230               11.0        1.000000   
              7           k95         1303               11.0        1.000000   
                          k_joint     1255               11.0        1.000000   
              8           k95         1302               11.0        1.000000   
                          k_joint     1273               11.0        1.000000   
validation    4           k95         1074               14.0        0.937500   
                          k_joint      783               14.0        1.000000   
              5           k95         1066               14.0        1.000000   
                          k_joint      898               14.0        1.000000   
              6           k95         1062               14.0        1.000000   
                          k_joint      944               14.0        1.000000   
              7           k95         1068               14.0        1.000000   
                          k_joint      983               14.0        1.000000   
              8           k95         1072               14.0        1.000000   
                          k_joint      996               14.0        1.000000   

                                      redundancia_mediana  
split         window_size point_type                       
test_internal 4           k95                    1.666667  
                          k_joint                1.800000  
              5           k95                    2.000000  
                          k_joint                2.000000  
              6           k95                    2.000000  
                          k_joint                2.272727  
              7           k95                    2.181818  
                          k_joint                2.181818  
              8           k95                    1.909091  
                          k_joint                1.909091  
validation    4           k95                    1.666667  
                          k_joint                1.736842  
              5           k95                    2.055556  
                          k_joint                2.181818  
              6           k95                    2.428571  
                          k_joint                2.500000  
              7           k95                    2.897933  
                          k_joint                2.937500  
              8           k95                    3.000000  
                          k_joint                3.000000

### 17. Catálogo de amostragem de todas as avaliações

A tabela confirma que cada combinação de split, modo e representação possui
amostras. Em seguida, uma mesma conversa de cada split é aberta nas sete
representações e nos dois modos (`frozen_probe` e `retrained`). Assim, o método
canônico, os blocos de quatro por interlocutor e cada sliding window 4–8 têm
exemplos diretamente criticáveis no notebook.


In [15]:
ctx_sample_catalog = (
    ctx_conversations[ctx_conversations.seed.eq(CTX_SEED)]
    .groupby(['split', 'model_mode', 'representation', 'prediction_category'], as_index=False)
    .agg(N=('sample_id', 'nunique'))
    .sort_values(['split', 'model_mode', 'representation', 'prediction_category'])
)
display(ctx_sample_catalog)

coverage_matrix = (
    ctx_conversations[ctx_conversations.seed.eq(CTX_SEED)]
    .groupby(['split', 'model_mode', 'representation']).sample_id.nunique()
    .unstack(['split', 'model_mode'])
)
display(coverage_matrix)


,split,model_mode,representation,prediction_category,N
0,test_internal,frozen_probe,canonical_message,FN,26
1,test_internal,frozen_probe,canonical_message,FP,46
2,test_internal,frozen_probe,canonical_message,TN,604
3,test_internal,frozen_probe,canonical_message,TP,714
4,test_internal,frozen_probe,chronological_window_4,FN,18
...,...,...,...,...,...
107,validation,retrained,chronological_window_8,TP,68
108,validation,retrained,speaker_block_4,FN,6
109,validation,retrained,speaker_block_4,FP,281
110,validation,retrained,speaker_block_4,TN,719


split                  test_internal             validation          
model_mode              frozen_probe retrained frozen_probe retrained
representation                                                       
canonical_message               1390      1390         1083      1083
chronological_window_4          1390      1390         1083      1083
chronological_window_5          1390      1390         1083      1083
chronological_window_6          1390      1390         1083      1083
chronological_window_7          1390      1390         1083      1083
chronological_window_8          1390      1390         1083      1083
speaker_block_4                 1390      1390         1083      1083

### 18. Amostras detalhadas: unidades e n-grams nas sete representações

In [16]:
CTX_SAMPLE_IDS = {'test_internal': None, 'validation': None}
CTX_SAMPLE_MODES = ('frozen_probe', 'retrained')

def choose_complete_sample(split):
    available = ctx_conversations[
        ctx_conversations.split.eq(split) & ctx_conversations.seed.eq(CTX_SEED)
    ].copy()
    expected = len(CTX_REPRESENTATIONS) * len(CTX_SAMPLE_MODES)
    complete = available.groupby('sample_id').size()
    complete_ids = set(complete[complete.eq(expected)].index.astype(str))
    preferred = available[
        available.model_mode.eq('retrained')
        & available.representation.eq('canonical_message')
        & available.prediction_category.eq('TP')
        & available.sample_id.astype(str).isin(complete_ids)
    ].sample_id.astype(str)
    candidates = sorted(set(preferred) or complete_ids)
    if not candidates:
        raise RuntimeError(f'Nenhuma conversa completa encontrada em {split}')
    return candidates[0]

for split in CTX_SAMPLE_IDS:
    if CTX_SAMPLE_IDS[split] is None:
        CTX_SAMPLE_IDS[split] = choose_complete_sample(split)

display(pd.DataFrame([
    {'split': split, 'sample_id': sample_id}
    for split, sample_id in CTX_SAMPLE_IDS.items()
]))

for split, sample_id in CTX_SAMPLE_IDS.items():
    display(Markdown(f'## Amostra `{split}` — `{sample_id}`'))
    metadata = ctx_conversations[
        ctx_conversations.split.eq(split)
        & ctx_conversations.seed.eq(CTX_SEED)
        & ctx_conversations.sample_id.astype(str).eq(str(sample_id))
    ].sort_values(['model_mode', 'representation'])
    display(metadata[[
        'model_mode', 'representation', 'prediction_category', 'label',
        'predicted_label', 'baseline_probability_scam', 'total_units',
        'total_original_messages',
    ]])

    for model_mode in CTX_SAMPLE_MODES:
        display(Markdown(f'### Modo `{model_mode}`'))
        for representation in CTX_REPRESENTATIONS:
            units = read_context_detail(
                'unit_effects', sample_id=sample_id, split=split,
                representation=representation, model_mode=model_mode,
                seed=CTX_SEED, columns=CTX_UNIT_COLUMNS,
            )
            spans = read_context_detail(
                'span_effects', sample_id=sample_id, split=split,
                representation=representation, model_mode=model_mode,
                seed=CTX_SEED, columns=CTX_SPAN_COLUMNS,
            )
            absolute = units.nlargest(6, 'abs_delta_logit').copy()
            supporting = units[units.class_support_delta.gt(1e-4)].nlargest(
                6, 'class_support_delta'
            ).copy()
            top_spans = spans[spans.class_support_delta.gt(1e-4)].nlargest(
                10, 'class_support_delta'
            ).copy()

            display(Markdown(
                f'#### `{representation}` — {len(units)} unidades; '
                f'{len(spans):,} perturbações de trecho'
            ))
            display(Markdown('**Top-6 por influência absoluta**'))
            display(absolute[[
                'block_index', 'message_model_indices', 'text', 'delta_logit',
                'class_support_delta', 'direction',
            ]])
            display(Markdown('**Até seis unidades que sustentam a decisão**'))
            display(supporting[[
                'block_index', 'message_model_indices', 'text',
                'class_support_delta', 'direction',
            ]])
            display(Markdown('**Top-10 n-grams que sustentam a decisão**'))
            display(top_spans[[
                'block_index', 'message_model_index', 'span_text', 'ngram_size',
                'delta_logit', 'class_support_delta', 'direction',
            ]])


,split,sample_id
0,test_internal,0057d286da968d6853f5bb87354aa1ecdf288f550a8a7cb0711ee47f8d18b586:0
1,validation,01564e3a279575e4011be01d95a7f90e2042f9ef32b601e579bcd3879d228fa1:0


## Amostra `test_internal` — `0057d286da968d6853f5bb87354aa1ecdf288f550a8a7cb0711ee47f8d18b586:0`

,model_mode,representation,prediction_category,label,predicted_label,baseline_probability_scam,total_units,total_original_messages
2949,frozen_probe,canonical_message,TP,1,1,0.985293,20,20
32625,frozen_probe,chronological_window_4,TP,1,1,0.992117,17,20
47463,frozen_probe,chronological_window_5,TP,1,1,0.991317,16,20
62301,frozen_probe,chronological_window_6,TP,1,1,0.989205,15,20
77139,frozen_probe,chronological_window_7,TP,1,1,0.986194,14,20
91977,frozen_probe,chronological_window_8,TP,1,1,0.984565,13,20
17787,frozen_probe,speaker_block_4,TP,1,1,0.913631,6,20
10368,retrained,canonical_message,TP,1,1,0.985293,20,20
40044,retrained,chronological_window_4,TP,1,1,0.982746,17,20
54882,retrained,chronological_window_5,TP,1,1,0.983156,16,20


### Modo `frozen_probe`

#### `canonical_message` — 20 unidades; 1,780 perturbações de trecho

**Top-6 por influência absoluta**

... (195 outputs left out)

**Até seis unidades que sustentam a decisão**

,block_index,message_model_indices,text,class_support_delta,direction


**Top-10 n-grams que sustentam a decisão**

,block_index,message_model_index,span_text,ngram_size,delta_logit,class_support_delta,direction
323,0,5,working in Poland. I,4,0.009033,0.009033,scam
379,0,5,working in Poland. I went,5,0.008835,0.008835,scam
266,0,5,working in Poland.,3,0.008235,0.008235,scam
102,0,3,everything is taken from,4,0.007641,0.007641,scam
109,0,3,when everything is taken from,5,0.007495,0.007495,scam
108,0,3,even when everything is taken,5,0.007465,0.007465,scam
110,0,3,everything is taken from them.,5,0.007385,0.007385,scam
322,0,5,was working in Poland.,4,0.007352,0.007352,scam
325,0,5,Poland. I went to,4,0.007103,0.007103,scam
997,1,8,country music like,3,0.006959,0.006959,scam


### 19. Curvas individuais das amostras nas sete representações

In [17]:
for split, sample_id in CTX_SAMPLE_IDS.items():
    for model_mode in CTX_SAMPLE_MODES:
        case = ctx_fidelity[
            ctx_fidelity.split.eq(split)
            & ctx_fidelity.model_mode.eq(model_mode)
            & ctx_fidelity.seed.eq(CTX_SEED)
            & ctx_fidelity.ranking_policy.eq(CTX_RANKING_POLICY)
            & ctx_fidelity.sample_id.astype(str).eq(str(sample_id))
        ]
        fig, axes = plt.subplots(1, 2, figsize=(19, 5))
        for representation, group in case.groupby('representation'):
            group = group.sort_values('unique_messages_covered')
            axes[0].plot(
                group.unique_messages_covered, group.comprehensiveness_logit_aligned,
                color=CTX_COLORS[representation], label=representation,
            )
            axes[1].plot(
                group.unique_messages_covered, group.sufficiency_distance_logit,
                color=CTX_COLORS[representation], label=representation,
            )
        axes[0].set_title(f'Comprehensiveness | {split} | {model_mode}')
        axes[1].set_title(f'Sufficiency distance | {split} | {model_mode}')
        for ax in axes:
            ax.set_xlabel('Mensagens originais únicas')
            ax.grid(alpha=.25); ax.legend(fontsize=7)
        plt.tight_layout(); plt.show()


/tmp/ipykernel_852356/1458262020.py:10: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  fig, axes = plt.subplots(1, 2, figsize=(19, 5))


## Checklist de interpretação

1. Compare sempre o mesmo split, modo, seed e população.
2. Leia comprehensiveness e sufficiency em conjunto.
3. Use boxplots para enxergar a heterogeneidade escondida pela média.
4. Controles aleatórios medem especificidade, não veracidade factual.
5. As amostras finais permitem auditar exatamente os blocos e n-grams usados em cada representação.
